In [74]:
# 그래프 관련
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
# matplotlib.rcParams['font.family'] = 'AppleGothic'
matplotlib.rcParams['axes.unicode_minus']=False

import pandas as pd
import numpy as np

# 머신러닝 관련
from sklearn import svm,metrics
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_validate
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, OneHotEncoder
from sklearn.metrics import mean_absolute_error 
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import plot_tree
from scipy.special import expit
from scipy.special import softmax
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import make_pipeline
import joblib

In [75]:
fish = pd.read_csv('data/fish_logistic.csv')
fish.head(2)

,Species,Weight,Length,Diagonal,Height,Width
0,Bream,242.0,25.4,30.0,11.52,4.0200
1,Bream,290.0,26.3,31.2,12.48,4.3056


In [76]:
data = fish[fish.columns[1:]].values
label = fish['Species'].values
train_data,test_data,train_label,test_label = train_test_split(data,label)
train_data[:2]

array([[975.    ,  41.    ,  45.9   ,  18.6354,   6.7473],
       [500.    ,  29.7   ,  34.5   ,  14.1795,   5.2785]])

In [77]:
# StandartScaler
ss = StandardScaler()
ss.fit(train_data)
train_scaled = ss.transform(train_data)
test_scaled = ss.transform(test_data)

In [78]:
# 데이터 학습
lr = LogisticRegression(C=20, max_iter=1000)
lr.fit(train_scaled, train_label)
predict = lr.predict(test_scaled)
print(f'학습 정확도: {lr.score(train_scaled, train_label)*100}%')
print(f'테스트 정확도: {lr.score(test_scaled, test_label)*100}%')

학습 정확도: 93.27731092436974%
테스트 정확도: 85.0%


In [79]:
#lr.predict_proba(test_scaled[:5])


#decision_function과 softmax를 조합하면 predict_proba를 직접 구현한 셈
decisions = lr.decision_function(test_scaled[:5])

# softmax 다중분류 : 0과 1숫자로 변환 확률 표현
soft1 = softmax(decisions,axis=1) #axis=1은 행방향
np.round(soft1, decimals=3)
print('train 정확도: ', lr.score(train_scaled, train_label))
print('test 정확도: ', lr.score(test_scaled, test_label))

train 정확도:  0.9327731092436975
test 정확도:  0.85


In [80]:
print(lr.classes_)
np.round(soft1,decimals=3)

['Bream' 'Parkki' 'Perch' 'Pike' 'Roach' 'Smelt' 'Whitefish']


array([[0.002, 0.013, 0.364, 0.003, 0.485, 0.   , 0.132],
       [0.   , 0.001, 0.056, 0.   , 0.004, 0.94 , 0.   ],
       [0.   , 0.049, 0.347, 0.001, 0.535, 0.057, 0.012],
       [0.041, 0.   , 0.624, 0.001, 0.   , 0.   , 0.334],
       [0.   , 0.019, 0.659, 0.001, 0.222, 0.092, 0.007]])

In [81]:
# 2. 파이프라인 생성 (다항 특성 -> 스케일러 -> 로지스틱 회귀)
model = make_pipeline(
    PolynomialFeatures(degree=2, include_bias=False),
    StandardScaler(),
    LogisticRegression(max_iter=1000)
)

# 3. 모델 학습 (파이프라인이 전처리와 학습을 한 번에 수행)
model.fit(train_data, train_label)

# 4. 성능 확인 (선택사항)
print(f'파이프라인 학습 정확도: {model.score(train_data, train_label)*100:.2f}%')
print(f'파이프라인 테스트 정확도: {model.score(test_data, test_label)*100:.2f}%')

# 5. 모델 저장
joblib.dump(model, 'model2.joblib')
print('모델 저장 완료')

파이프라인 학습 정확도: 83.19%
파이프라인 테스트 정확도: 77.50%
모델 저장 완료
